# IndoML Datathon — Track 1: Noise Event Detection (Vaani) — multi-GPU

**Frozen BEATs + FDY-CRNN, mean-teacher semi-supervision on a per-tier masked loss,
frequency MixStyle for domain shift, cSEBBs post-processing.**

This variant is for a **multi-GPU box** - Kaggle's `GPU T4 x2` accelerator, or any
Colab runtime that actually hands you more than one GPU. Training runs under
`torch.nn.parallel.DistributedDataParallel` via `torchrun`, one process per GPU, so it
scales close to linearly instead of the ~1.3x a naive `DataParallel` would give an
RNN-heavy model like this one.

No Google Drive here: Kaggle does not offer a Drive mount, so this notebook keeps
everything on the machine's own local disk and downloads the dataset fresh each
session. If you want the dataset to survive a session restart on Kaggle, save the
downloaded `$DATA` folder as a **Kaggle Dataset** (Add Data -> New Dataset -> your
output) and attach it as an input next time; that's the Kaggle equivalent of the
Drive cache in the single-GPU Colab notebook, but it's a manual step this notebook
does not automate.

| Step | Cell | Notes |
|---|---|---|
| 0 | GPU check | detects however many GPUs this session actually has |
| 1 | Clone + install | never installs torch (the platform's is CUDA-linked) |
| 2 | HF login | the dataset is gated |
| 3 | Download data | straight to local disk, no Drive |
| 4 | BEATs weights | ~360 MB, local disk |
| 5 | Train | `torchrun`, one process per GPU |
| 6 | Tune cSEBBs | no retraining, biggest ROI per minute |
| 7 | Predict | writes `submission.zip` (`predictions.jsonl`) |
| 8 | Validate | checks the archive against the Codabench format |


## 0 · GPU check


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), 'No GPU! Pick a GPU runtime/accelerator first.'
NGPU = torch.cuda.device_count()
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| GPUs visible:', NGPU)
for i in range(NGPU):
    print('  -', i, torch.cuda.get_device_name(i))
if NGPU < 2:
    print('\nOnly 1 GPU visible - the training cell below still works, it just runs '
          'single-process instead of DDP. On Kaggle, pick the "GPU T4 x2" accelerator '
          '(Settings -> Accelerator) to get both T4s.')


## 1 · Clone the repo and install dependencies

`requirements.txt` deliberately excludes torch: both platforms ship a CUDA-linked
build already, and pip-installing torch here would replace it with a mismatched wheel
and silently kill the GPU.

Works on both platforms without edits: Kaggle's writable directory is `/kaggle/working`,
Colab's is `/content` - this cell picks whichever exists.


In [ ]:
import os, sys

REPO = 'https://github.com/raut7218/vaani-sed-track1.git'
ROOT = '/kaggle/working' if os.path.isdir('/kaggle') else '/content'
REPO_DIR = f'{ROOT}/vaani-sed-track1'

if not os.path.exists(REPO_DIR):
    !git clone -q $REPO $REPO_DIR
%cd $REPO_DIR
!git pull -q || true

# everything except torch/torchaudio, which the platform already provides
!pip install -q soundfile librosa 'datasets>=2.18' huggingface_hub pyyaml tqdm
sys.path.insert(0, REPO_DIR)

DATA = f'{ROOT}/work/data'
RUN = f'{ROOT}/work/runs/baseline'
BEATS_DIR = f'{ROOT}/work/checkpoints'
for d in (DATA, RUN, BEATS_DIR):
    os.makedirs(d, exist_ok=True)
print('ROOT =', ROOT)
print('DATA =', DATA)
print('RUN  =', RUN)
print('BEATS_DIR =', BEATS_DIR)


## 2 · Hugging Face login (the dataset is gated)

`ARTPARK-IISc/Vaani-Noise-Event-Dataset` is gated, so every file request needs a token
from an account that has been granted access.

1. Open the [dataset page](https://huggingface.co/datasets/ARTPARK-IISc/Vaani-Noise-Event-Dataset)
   and click **Agree and access**.
2. Make a token at [hf.co/settings/tokens](https://huggingface.co/settings/tokens) (read scope).
3. Add it as a secret named `HF_TOKEN`:
   - **Colab**: Secrets (key icon, left sidebar) -> add `HF_TOKEN` -> enable **Notebook access**.
   - **Kaggle**: Add-ons -> Secrets -> add `HF_TOKEN` -> attach it to this notebook.

The scripts find that secret on their own, on either platform. The cell below just
confirms it works.


In [ ]:
from huggingface_hub import HfApi
from scripts.download_data import resolve_token, REPO

tok = resolve_token()
assert tok, 'No HF token found. Add HF_TOKEN as a Colab/Kaggle secret for this notebook.'

who = HfApi(token=tok).whoami()
print('logged in as:', who.get('name'))
try:
    HfApi(token=tok).repo_info(REPO, repo_type='dataset', files_metadata=False)
    # a real file request is what gating actually blocks
    from huggingface_hub import hf_hub_download
    hf_hub_download(REPO, 'README.md', repo_type='dataset', token=tok)
    print('access to', REPO, 'CONFIRMED')
except Exception as e:
    print('NO ACCESS yet:', e)
    print("-> click 'Agree and access' on the dataset page, then re-run this cell")


## 3 · Download the dataset

**182 shards, 16.5 GB of parquet, 90,637 clips (~154.6 h).** Decoded to FLAC that is
roughly **9 GB** - comfortable on either platform's local disk. Straight to `$DATA`,
no Drive involved.

Shards are fetched one at a time and each parquet blob is deleted once its clips are
written, so peak disk stays near the decoded size rather than decoded + 16.5 GB.
Downloads resume by clip, so a re-run after a disconnect only fetches what is missing -
but on Kaggle a *new* session starts with an empty `$DATA` again unless you saved last
session's output as a Kaggle Dataset and attached it as an input (see the intro cell).


See what is on the server (downloads nothing):


In [ ]:
!python scripts/download_data.py --out $DATA --list-only


**Start small.** Two shards is ~1000 clips - enough to confirm the whole pipeline end
to end in a few minutes before committing to the full download.


In [ ]:
!python scripts/download_data.py --out $DATA --max-shards 2

import json
print(json.dumps(json.load(open(f'{DATA}/stats.json')), indent=2)[:2500])


Then the full corpus. This is the long one - expect a while for 16.5 GB plus decode.
It skips whatever the previous cell already wrote.


In [ ]:
!python scripts/download_data.py --out $DATA


### Check the tier split

The full corpus has an **`annotationQuality`** column, which is the gold/silver/bronze
signal the sample dataset lacked - so tiers are read from the data rather than assumed.

The download prints every value it saw. If any value could not be mapped it says so
loudly: add it to `QUALITY_ALIASES` in `src/data/prepare.py` and re-run (already
materialised clips are skipped, so it only rewrites the manifest and is quick).

A clip labelled gold/silver but carrying no timestamps is demoted to bronze - without
timestamps there is nothing for the frame-level loss to consume.


In [ ]:
import json, collections
recs = [json.loads(l) for l in open(f'{DATA}/manifest.jsonl', encoding='utf-8')]
tier_h = collections.Counter()
for r in recs: tier_h[r['tier']] += r['duration']
print('clips per tier :', dict(collections.Counter(r['tier'] for r in recs)))
print('hours per tier :', {k: round(v/3600, 2) for k, v in tier_h.items()})
print('states         :', len({r['state'] for r in recs}),
      '| languages:', len({r['language'] for r in recs}))
st = json.load(open(f'{DATA}/stats.json'))
print('annotationQuality values seen:', st.get('annotationQuality_values_seen'))
print('UNMAPPED (fix these!)        :', st.get('unmapped_annotationQuality'))


## 4 · BEATs checkpoint (~360 MB)


In [ ]:
from src.models.beats_encoder import download_beats
p = download_beats(BEATS_DIR)
print('BEATs at:', p)


## 5 · Train

One model, all three tiers. Every batch mixes gold/silver/bronze so mean-teacher
consistency and per-tier MixStyle always have material to work with.

Launched with `torchrun`: one process per visible GPU, gradients synced across them
via `DistributedDataParallel`. `--batch-size` below is **per GPU**, matching the
standard DDP convention - `PER_GPU_BS * NGPU` is the effective global batch. This cell
keeps the global batch equal to the single-GPU notebook's default of 24 (so runs are
comparable): 12/GPU x 2 GPUs = 24. Raise `PER_GPU_BS` for more throughput if you don't
need that comparability, and start lower if either T4 goes OOM.

With `NGPU == 1` this still works - `torchrun --nproc_per_node=1` is just a single
process, equivalent to running `python -m src.train.train` directly.


In [ ]:
import yaml

TOTAL_BATCH = 24
PER_GPU_BS = max(1, TOTAL_BATCH // NGPU)
print('GPUs:', NGPU, '| per-GPU batch:', PER_GPU_BS, '| global batch:', PER_GPU_BS * NGPU)

run_cfg = yaml.safe_load(open('configs/default.yaml'))
run_cfg['model']['beats_dir'] = BEATS_DIR
run_cfg['model']['beats_ckpt'] = str(p) if p else ''
RUN_CFG = f'{ROOT}/work/config_run.yaml'
yaml.safe_dump(run_cfg, open(RUN_CFG, 'w'))

!torchrun --standalone --nproc_per_node=$NGPU -m src.train.train \
    --config $RUN_CFG \
    --data $DATA \
    --out $RUN \
    --epochs 30 \
    --batch-size $PER_GPU_BS


### Training curve


In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open(f'{RUN}/history.json'))
fig, ax = plt.subplots(figsize=(7,4))
for which in ('student','teacher'):
    xs = [r['epoch'] for r in h if r['which']==which]
    ys = [r['score'] for r in h if r['which']==which]
    if xs: ax.plot(xs, ys, marker='o', label=which)
ax.set_xlabel('epoch'); ax.set_ylabel('0.5*F1 + 0.5*Dice'); ax.legend(); ax.grid(alpha=.3)
plt.show()
print('best:', max((r['score'] for r in h), default=None))


## 6 · Tune the post-processor

Runs on cached validation scores - **no retraining**, and single-process (tuning is
cheap enough that DDP would be pure overhead here). This is the cheapest large win
available; it also prints the plain median-filter baseline so you can see the delta.


In [ ]:
!python scripts/tune_postproc.py --run $RUN --rounds 2


## 7 · Predict → `submission.zip`

Point `--audio-dir` at the released test audio. Output is class-agnostic
onset/offset pairs, which is what Track 1 is scored on. Inference is single-GPU -
there is no benefit to DDP for a forward-only pass over the test set.

The archive is what you upload to
[Codabench competition 17825](https://www.codabench.org/competitions/17825/):
a ZIP holding a single `predictions.jsonl` at its root, one JSON object per clip.


In [ ]:
TEST_AUDIO = f'{ROOT}/test_audio'   # <-- point at the official test set

import os
if os.path.isdir(TEST_AUDIO) and os.listdir(TEST_AUDIO):
    !python -m src.infer.predict \
        --ckpt $RUN/best.pt \
        --audio-dir $TEST_AUDIO \
        --params $RUN/postproc_params.json \
        --out submission.zip
else:
    print('No test audio yet - running on the training manifest as a demo instead.')
    !python -m src.infer.predict \
        --ckpt $RUN/best.pt \
        --manifest $DATA/manifest.jsonl \
        --params $RUN/postproc_params.json \
        --out submission.zip


In [ ]:
# Validate the archive against the competition's stated format before uploading.
import json, zipfile

with zipfile.ZipFile('submission.zip') as z:
    assert z.namelist() == ['predictions.jsonl'], z.namelist()
    lines = z.read('predictions.jsonl').decode('utf-8').strip().split('\n')

seen, n_ev = set(), 0
for line in lines:
    rec = json.loads(line)
    assert set(rec) == {'clip_id', 'events'}, rec.keys()
    assert rec['clip_id'] not in seen, 'duplicate clip_id ' + rec['clip_id']
    seen.add(rec['clip_id'])
    for ev in rec['events']:
        assert set(ev) == {'onset', 'offset'}
        assert 0.0 <= ev['onset'] <= ev['offset'], ev
    n_ev += len(rec['events'])

print('OK:', len(seen), 'clips,', n_ev, 'events')
for line in lines[:3]:
    print(line[:160])


### Get the submission

On Colab this downloads it straight to your machine. On Kaggle, files written to
`/kaggle/working` are already picked up as notebook Output - no download call needed,
just check the **Output** tab (or **Save Version** to persist it), the cell below
only prints where it landed.


In [ ]:
import os
if os.path.isdir('/kaggle'):
    print('On Kaggle: submission.zip is at', os.path.abspath('submission.zip'))
    print('It will show up under this notebook\'s Output tab once you save a version.')
else:
    from google.colab import files
    files.download('submission.zip')


---
## Where to go next

The build order in the README is designed so you are always shippable:

1. ✅ this notebook = baseline + cSEBBs, running on 2 GPUs
2. **Frequency MixStyle sweep** — `model.mixstyle_p` in the config (0.3 / 0.5 / 0.7)
3. **Self-training** — pseudo-label the bronze tier with this model, promote confident
   predictions to silver, retrain. See `README.md`.
4. **Seed ensembling** — train 3 seeds, average frame scores, re-tune cSEBBs on the
   ensemble (re-tuning after ensembling matters; the score distribution shifts).

**Ablations worth running** (each is one config flag):

| Flag | Tests |
|---|---|
| `--no-beats` | how much BEATs is actually worth on Vaani |
| `model.n_basis: 1` | FDY vs plain CRNN — check per class, it can hurt fans/engines |
| `loss.lambda_cons: 0` | value of mean-teacher |
| `--method median` in tuning | cSEBBs vs frame thresholding |
